# Product Coding Agent — single scrape artifact

Codes requested product features by navigating the scrape artifact folder and reusing the existing LLM service.

In [ ]:
%config Completer.use_jedi = False

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Make notebook execution stable even before the dedicated kernel is registered.
os.environ.setdefault('PYTHONPATH', str(SRC))
os.environ.setdefault('PCT_IPYTHON_DISABLE_JEDI', 'true')
# Default to direct HTTP LLM transport; avoids broken OpenAI SDK imports in AzureML notebooks.
os.environ.setdefault('PCT_LLM_TRANSPORT', 'httpx')
PROJECT_ROOT


In [ ]:
from product_coding_tool import CodingRequest, FeatureRule, ProductCodingAgent
from product_coding_tool.log import setup_logging

setup_logging('INFO')


In [ ]:
artifact_dir = PROJECT_ROOT / 'data' / 'scraped' / 'scrape_20260628_190357_25c16d76'

features = [
    FeatureRule(feature_id='11707', feature_name='BRAND', feature_type='open_set', definition='Brand under which the toy is sold.', aliases=['brand name']),
    FeatureRule(feature_id='227098', feature_name='TOY MANUFACTURER', feature_type='open_set', definition='Manufacturer or producer of the toy product.', aliases=['manufacturer', 'producer']),
    FeatureRule(feature_id='BATTERY_REQUIRED', feature_name='Battery Required', feature_type='closed_set', definition='Whether the toy requires batteries.', allowed_values=['Yes', 'No', 'Not stated'], evidence_hints=['battery', 'batteries', 'AA', 'AAA'])
]

request = CodingRequest(artifact_dir=artifact_dir, features=features)
result = ProductCodingAgent().run(request)
result


In [ ]:
import pandas as pd
rows = [r.model_dump(mode='json', exclude={'evidence', 'audit'}) for r in result.results]
pd.DataFrame(rows)
